## Settings

In [ ]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [ ]:
## libraries
import sys
import pickle
import logging
from pathlib import Path
from IPython.display import Markdown, display

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import (
    load_processed_data, 
    load_falsified_data
)
from src.estimators.factories import load_estimators
from src.evaluators.caching import load_notebook_cache
from src.evaluators.tables import main_table
from src.evaluators.falsifying import (
    train_falsified_transfer,
    train_falsified_agreement,
    train_falsified_consensus,
    compile_falsified_transfer,
    compile_falsified_agreement,
    compile_falsified_consensus,
    compile_falsified_full,
    stat_falsified_test,
    stat_falsified_summary
)

## constants
from src.evaluators.config import (
    FRONTIER_METRICS,
    CONSENSUS_METRICS
)
from src.evaluators.config import (
    FEAT_X, 
    FEAT_Z, 
    TARGET
)

## Initialization

In [ ]:
## reproducibility
N_DECIMALS = 2
N_REPEATS = 30
RANDOM_STATE = 42
FORCE_RECOMPUTE = False

## load data and models
_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    data_proc = load_processed_data()
    data_fals = load_falsified_data()
finally:
    logging.disable(_disable)
    models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data_proc.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")
print("Falsified Data:")
for method, data in data_fals.items():
    n_r, n_c = data.shape
    print(f" - {method}: {n_c} features, {n_r} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

## cache setup
cache_path = root / "notebooks" / "cache" / "falsify_results.pkl"
cache_metadata = {
    "n_obs": len(data_proc),
    "n_repeats": N_REPEATS,
    "random_state": RANDOM_STATE,
    "model_names": sorted(models.keys()),
    "target": TARGET,
    "feat_x": list(FEAT_X),
    "feat_z": list(FEAT_Z),
}
cache_keys = (
    "results_falsified_transfer",
    "results_falsified_agreement",
    "results_falsified_consensus",
)
cache_payload = load_notebook_cache(
    cache_path = cache_path,
    metadata = cache_metadata,
    required_keys = cache_keys,
    force_recompute = FORCE_RECOMPUTE,
)
if cache_payload is not None:
    globals().update({key: cache_payload[key] for key in cache_keys})

Original Data: 32 features, 25 observations
Falsified Data:
 - random_generate: 32 features, 25 observations
 - target_remap: 32 features, 25 observations
 - vector_generate: 32 features, 25 observations
Learned Models: 9 estimators
Loaded results from /Users/nickkunz/Documents/Nick’s MacBook Pro/Python/network-capacity/notebooks/cache/falsify_results.pkl


## Training

In [ ]:
## train falsified transfer
if cache_payload is None:
    results_dict_falsified_transfer = train_falsified_transfer(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        group = "domain",
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE
    )

In [ ]:
## train falsified prediction consensus
if cache_payload is None:
    results_dict_falsified_agreement = train_falsified_agreement(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
    )

In [ ]:
## train falsified pairwise consensus
if cache_payload is None:
    results_dict_falsified_consensus = train_falsified_consensus(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE
    )

## Post-Processing

In [ ]:
if cache_payload is None:
    results_falsified_transfer = compile_falsified_transfer(
        results = results_dict_falsified_transfer,
    )
    results_falsified_agreement = compile_falsified_agreement(
        results = results_dict_falsified_agreement,
    )

if cache_payload is None or "results_falsified_full_agreement" not in cache_payload:
    if cache_payload is not None:
        results_dict_falsified_consensus = train_falsified_consensus(
            data_proc = data_proc,
            data_fals = data_fals,
            models = models,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            target = TARGET,
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE,
            n_jobs = -1,
        )
    results_falsified_consensus = compile_falsified_consensus(
        results = results_dict_falsified_consensus,
    )
    results_falsified_full_agreement = compile_falsified_full(
        results = results_dict_falsified_consensus,
        data_proc = data_proc,
        data_fals = data_fals,
        target = TARGET,
    )
    cache_path.parent.mkdir(parents = True, exist_ok = True)
    cache_payload = {
        **(cache_payload or {}),
        "metadata": cache_metadata,
        "results_falsified_transfer": results_falsified_transfer,
        "results_falsified_agreement": results_falsified_agreement,
        "results_falsified_consensus": results_falsified_consensus,
        "results_falsified_full_agreement": results_falsified_full_agreement,
    }
    cache_path.write_bytes(pickle.dumps(cache_payload))
    print(f"Cached falsification results -> {cache_path}")

## Domain Transfer Difference Test
The transfer falsifiability test probes whether deliberately corrupted data reduce EI transfer performance relative to the original empirical baseline. Three falsification types (target remap, random generate, vector generate) are evaluated under frozen and retrained LOGO-CV (domain) protocols and scored by EI per (model x domain). Each falsification method is tested with a paired one-sided Wilcoxon signed-rank statistic comparing falsified and original EI values, with Δ defined as falsified - original.

- **$H_0$**: Falsified EI does not fall below original EI ($\Delta EI \ge 0$).
- **$H_1$**: Falsified EI falls below original EI ($\Delta EI < 0$).

Rejecting $H_0$ establishes that EI transfer performance is significantly weaker on falsified data than on original data.

### Falsification Protocol
Target remap corrupts the target association, while random and vector generation corrupt the input feature geometry. Under the frozen protocol, models trained on original data are evaluated directly on falsified inputs. Under the retrain protocol, models are refit on falsified data and compared back to the original baseline.

### Empirical Test Specification
Original and falsified EI values are paired on (model $\times$ domain) within each falsification method and protocol. The paired median shift, rank-biserial effect size, and Holm-adjusted one-sided p-value summarize whether each destructive intervention produces a consistent EI loss.

In [ ]:
## wilcoxon signed-rank test for falsified transfer
results_table_transfer = stat_falsified_test(
    results = results_falsified_transfer,
    feat_value = ["ei"],
    feat_pairs = ["model", "group"],
    decimals = N_DECIMALS
)

display(results_table_transfer)

Paired One-Sided Test (Wilcoxon Signed-Rank): n = 45
H₀: Δ EI ≥ 0
H₁: Δ EI < 0
Median Δ EI: Median of paired differences (falsified - original), not the difference of marginal medians
Rank-biserial r: Raw paired effect size; negative values indicate falsified < original
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Diff.: Yes if Holm-adj. p < 0.05 and Median Δ < 0
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


Median Δ EI Rank-biserial r One-sided p  \
Falsification Method                                                    
frozen        target remap          -0.03           -0.44       0.005   
              random generate       -0.04           -0.98      <0.001   
              vector generate       -0.03           -0.80      <0.001   
retrain       target remap          -0.08           -0.57      <0.001   
              random generate       -0.02           -0.71      <0.001   
              vector generate       -0.03           -0.67      <0.001   

                              Holm-adj. p Sig. Diff.  
Falsification Method                                  
frozen        target remap          0.005   **   Yes  
              random generate      <0.001  ***   Yes  
              vector generate      <0.001  ***   Yes  
retrain       target remap         <0.001  ***   Yes  
              random generate      <0.001  ***   Yes  
              vector generate      <0.001  ***   Yes

In [ ]:
## median falsified transfer metrics
transfer_summary = stat_falsified_summary(
    results = results_falsified_transfer,
    metrics = FRONTIER_METRICS,
    decimals = 2
)

display(transfer_summary)

EI [IQR]   VR   MV    MS
Falsification Method                                            
original      original         0.76 [0.72, 0.81]  0.0  0.0  5.84
frozen        target remap     0.73 [0.71, 0.78]  0.0  0.0  6.23
              random generate  0.72 [0.67, 0.76]  0.0  0.0  8.93
              vector generate  0.74 [0.68, 0.76]  0.0  0.0  7.57
retrain       target remap     0.71 [0.64, 0.77]  0.0  0.0  6.42
              random generate  0.75 [0.70, 0.76]  0.0  0.0  7.32
              vector generate  0.73 [0.68, 0.76]  0.0  0.0  6.69

_All three falsification types significantly reduced frontier efficiency under both protocols after Holm correction. Frozen falsifications produced median Δ EI of −0.03 for target permutation, −0.04 for random generation, and −0.03 for vector generation (all Holm-adjusted p ≤ 0.005). After retraining, median Δ EI was −0.08 for target permutation, −0.02 for random generation, and −0.03 for vector generation (all Holm-adjusted p < 0.001). Degrading the representation–target correspondence or the feature structure thus consistently impairs frontier efficiency._


## Prediction Consensus Difference Test
The prediction consensus falsifiability test probes whether falsified data produce weaker agreement between frontier predictions and observed capacity ordering than original data. Each falsification method is tested with a paired one-sided Wilcoxon signed-rank statistic comparing falsified and original CI values, with Δ defined as falsified - original.

- **$H_0$**: Falsified CI does not fall below original CI ($\Delta CI \ge 0$).
- **$H_1$**: Falsified CI falls below original CI ($\Delta CI < 0$).

Rejecting $H_0$ establishes that prediction consensus is significantly weaker on falsified data than on original data.

### Falsification Protocol
Each falsification method is treated as an independent destructive intervention on the signal used for consensus. Frozen contrasts evaluate whether original model fits retain consensus when scored on falsified data. Retrained contrasts evaluate whether refitting on falsified data restores consensus with the original-data baseline.

### Empirical Test Specification
Original and falsified CI values are paired on (model $\times$ domain) within each falsification method and protocol. The paired median shift, rank-biserial effect size, and Holm-adjusted one-sided p-value summarize whether each intervention produces a consistent loss of prediction consensus.

In [ ]:
## wilcoxon signed-rank test for falsified prediction consensus
results_table_agreement = stat_falsified_test(
    results = results_falsified_agreement,
    feat_value = ["ci"],
    feat_pairs = ["model", "group"],
    decimals = N_DECIMALS
)

display(results_table_agreement)

Paired One-Sided Test (Wilcoxon Signed-Rank): n = 45
H₀: Δ CI ≥ 0
H₁: Δ CI < 0
Median Δ CI: Median of paired differences (falsified - original), not the difference of marginal medians
Rank-biserial r: Raw paired effect size; negative values indicate falsified < original
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Diff.: Yes if Holm-adj. p < 0.05 and Median Δ < 0
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


Median Δ CI Rank-biserial r One-sided p  \
Falsification Method                                                    
frozen        target remap          -0.29           -1.00      <0.001   
              random generate       -0.30           -0.90      <0.001   
              vector generate       -0.21           -0.95      <0.001   
retrain       target remap          -0.25           -0.99      <0.001   
              random generate       -0.17           -0.94      <0.001   
              vector generate       -0.31           -0.99      <0.001   

                              Holm-adj. p Sig. Diff.  
Falsification Method                                  
frozen        target remap         <0.001  ***   Yes  
              random generate      <0.001  ***   Yes  
              vector generate      <0.001  ***   Yes  
retrain       target remap         <0.001  ***   Yes  
              random generate      <0.001  ***   Yes  
              vector generate      <0.001  ***   Yes

In [ ]:
## median falsified prediction consensus metrics
agreement_summary = stat_falsified_summary(
    results = results_falsified_agreement,
    metrics = CONSENSUS_METRICS,
    decimals = N_DECIMALS
)

display(agreement_summary)

CI [IQR]    ρ   RBO   DCR
Falsification Method                                             
original      original         0.84 [0.77, 0.92]  0.7  0.86  0.79
frozen        target remap     0.53 [0.49, 0.63] -0.3  0.78  0.59
              random generate  0.52 [0.46, 0.64] -0.3  0.76  0.64
              vector generate  0.63 [0.53, 0.68]  0.1  0.83  0.55
retrain       target remap     0.54 [0.49, 0.61] -0.3  0.78  0.66
              random generate  0.64 [0.56, 0.71]  0.0  0.81  0.62
              vector generate  0.52 [0.46, 0.60] -0.3  0.78  0.60

_Every falsification type significantly reduced prediction consensus under both protocols after Holm correction. Median Δ CI ranged from −0.30 to −0.21 under the frozen protocol and from −0.31 to −0.17 after retraining, with rank-biserial effects from −0.90 to −1.00 (all Holm-adjusted p < 0.001). Retraining therefore did not restore prediction consensus for any falsification type._


## Pairwise Prediction Consensus Evaluation
The pairwise prediction consensus evaluation probes whether falsified inputs preserve agreement among independently trained frontiers. Unlike the transfer and prediction-consensus sections, this is a descriptive stability analysis rather than a formal falsifiability test. Agreement is computed across all model pairs using the same consensus metrics.

- **Spearman (`ρ`)**: Global rank agreement between paired model frontiers.
- **Rank-Biased Overlap (`RBO`)**: Top-weighted agreement, emphasizing the high-capacity tail.
- **Distance Correlation (`dCor`)**: Nonlinear dependence between paired model frontiers.
- **Consensus Index (`CI`)**: Geometric mean of agreement-scaled `ρ`, `RBO`, and `dCor`.

### Falsification Protocol
The evaluation is restricted to the frozen protocol, so model fits remain anchored to the original empirical manifold while falsified inputs are scored. Only falsified-input rows enter the preview and visualization; the original baseline remains available in the compiled results for comparison.

### Reporting Convention
Each falsification method contributes one value per model pair under the frozen protocol. The preview reports medians across all retained frozen falsified pairwise comparisons. Prior to compounding, `ρ` is rescaled from `[-1, 1]` to `[0, 1]` so that negative rank agreement lowers `CI` without collapsing to a single floor. Higher values indicate stronger falsification-stable pairwise prediction consensus.

In [ ]:
## consensus preview
results_falsified_consensus_fals = results_falsified_consensus.loc[
    (results_falsified_consensus["Falsification"] == "frozen")
    & (results_falsified_consensus["condition"] == "falsified")
].copy()

round(number = results_falsified_consensus_fals[CONSENSUS_METRICS].median(), ndigits = N_DECIMALS)

rho    0.68
rbo    0.65
dcr    0.67
ci     0.70
dtype: float64

_Frozen falsification reduces pairwise prediction consensus for generated-input interventions, with median `CI` = 0.70 across retained pairwise comparisons. Target remap preserves frozen consensus by construction because input features are unchanged, whereas random and vector generation visibly weaken agreement among fitted frontiers. Pairwise prediction consensus therefore depends on coherent event-network geometry rather than arbitrary or generated input structure._


## Summary

In [ ]:
## manuscript master table for falsification validity
display_table = main_table(
    results_left = results_table_transfer.reset_index(),
    results_right = results_table_agreement.reset_index(),
    index_feat = ["Falsification", "Method"],
    value_feat_left = ["Median Δ EI", "Rank-biserial r", "Sig.", "Diff."],
    value_feat_right = ["Median Δ CI", "Rank-biserial r", "Sig.", "Diff."],
    header_left = "Domain Transfer",
    header_right = "Prediction Consensus",
    short_feat_left = ["Δ EI", "r", "Sig.", "Diff."],
    short_feat_right = ["Δ CI", "r", "Sig.", "Diff."],
    row_order = [
        ("frozen",  "target remap"), ("frozen",  "random generate"), ("frozen",  "vector generate"),
        ("retrain", "target remap"), ("retrain", "random generate"), ("retrain", "vector generate"),
    ],
)

display(display_table)
display(
    Markdown(
    "**Δ EI**, **Δ CI**: median signed difference between falsified and original performance, where negative values indicate falsified data underperform original data. "
    "**r**: rank-biserial correlation effect size, where negative values favor original > falsified. "
    "**Sig.**: Holm-adjusted one-sided Wilcoxon *p* below threshold to test within each outcome "
    "(\*\*\* *p* < 0.001, \*\* *p* < 0.01, \* *p* < 0.05); significance supports the difference hypothesis "
    "H₁: Δ < 0, that falsified data underperform original data. "
    "**Diff.**: difference supported when Holm-adjusted *p* < 0.05 and median Δ < 0. "
    "Each test is conducted on *n* = 45 (9 models × 5 domains)."
    )
)

Domain Transfer                    \
                                         Δ EI      r Sig. Diff.   
Falsification Method                                              
frozen        target remap              -0.03  -0.44   **   Yes   
              random generate           -0.04  -0.98  ***   Yes   
              vector generate           -0.03  -0.80  ***   Yes   
retrain       target remap              -0.08  -0.57  ***   Yes   
              random generate           -0.02  -0.71  ***   Yes   
              vector generate           -0.03  -0.67  ***   Yes   

                              Prediction Consensus                    
                                              Δ CI      r Sig. Diff.  
Falsification Method                                                  
frozen        target remap                   -0.29  -1.00  ***   Yes  
              random generate                -0.30  -0.90  ***   Yes  
              vector generate                -0.21  -0.95  ***   Yes  
retrain       target remap                   -0.25  -0.99  ***   Yes  
              random generate                -0.17  -0.94  ***   Yes  
              vector generate                -0.31  -0.99  ***   Yes

**Δ EI**, **Δ CI**: median signed difference between falsified and original performance, where negative values indicate falsified data underperform original data. **r**: rank-biserial correlation effect size, where negative values favor original > falsified. **Sig.**: Holm-adjusted one-sided Wilcoxon *p* below threshold to test within each outcome (\*\*\* *p* < 0.001, \*\* *p* < 0.01, \* *p* < 0.05); significance supports the difference hypothesis H₁: Δ < 0, that falsified data underperform original data. **Diff.**: difference supported when Holm-adjusted *p* < 0.05 and median Δ < 0. Each test is conducted on *n* = 45 (9 models × 5 domains).